# 02 · Baseline Model
## LLM Classification Finetuning · Kaggle Competition

Modelo de referencia usando features numéricas y TF-IDF.
Estrategia de validación: GroupKFold por prompt para evitar data leakage.

In [1]:
import os
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss
from scipy.sparse import hstack, csr_matrix

import warnings
warnings.filterwarnings('ignore')

BASE_PATH = '/kaggle/input/competitions/llm-classification-finetuning'
train = pd.read_csv(f'{BASE_PATH}/train.csv')
test  = pd.read_csv(f'{BASE_PATH}/test.csv')
sub   = pd.read_csv(f'{BASE_PATH}/sample_submission.csv')

print(f'Train: {train.shape} | Test: {test.shape}')

Train: (57477, 9) | Test: (3, 4)


In [2]:
def extract_last_turn(text):
    """Extrae el último turno de una conversación serializada."""
    try:
        parsed = json.loads(text)
        return parsed[-1] if parsed else text
    except:
        try:
            cleaned = re.sub(r'\\(?!["\\/bfnrt]|u[0-9a-fA-F]{4})', r'\\\\', text)
            parsed = json.loads(cleaned)
            return parsed[-1] if parsed else text
        except:
            return text

# Extraer último turno
for df in [train, test]:
    df['prompt_clean']     = df['prompt'].apply(extract_last_turn)
    df['response_a_clean'] = df['response_a'].apply(extract_last_turn)
    df['response_b_clean'] = df['response_b'].apply(extract_last_turn)

print("Preprocesamiento completado")
print(train['prompt_clean'].iloc[0][:200])

Preprocesamiento completado
OK, does pineapple belong on a pizza? Relax and give me fun answer.


In [3]:
def add_length_features(df):
    df['len_prompt']     = df['prompt_clean'].str.len()
    df['len_response_a'] = df['response_a_clean'].str.len()
    df['len_response_b'] = df['response_b_clean'].str.len()
    df['len_diff']       = df['len_response_a'] - df['len_response_b']
    df['len_ratio']      = df['len_response_a'] / (df['len_response_b'] + 1)
    df['a_is_longer']    = (df['len_response_a'] > df['len_response_b']).astype(int)
    return df

train = add_length_features(train)
test  = add_length_features(test)

numeric_features = ['len_prompt', 'len_response_a', 'len_response_b',
                    'len_diff', 'len_ratio', 'a_is_longer']

print(train[numeric_features].describe().round(1))

       len_prompt  len_response_a  len_response_b  len_diff  len_ratio  \
count     57477.0         57410.0         57409.0   57386.0    57386.0   
mean        298.9          1084.5          1089.6      -5.3        2.8   
std         907.6           873.3           882.9     779.6       38.0   
min           1.0             0.0             0.0  -19864.0        0.0   
25%          42.0           360.0           363.0    -377.0        0.6   
50%          76.0           929.5           935.0       0.0        1.0   
75%         181.0          1614.0          1625.0     370.0        1.6   
max       12000.0          7961.0         21443.0    6155.0     4824.0   

       a_is_longer  
count      57477.0  
mean           0.5  
std            0.5  
min            0.0  
25%            0.0  
50%            0.0  
75%            1.0  
max            1.0  


In [4]:
# Target
def get_winner(row):
    if row['winner_model_a'] == 1: return 'model_a'
    if row['winner_model_b'] == 1: return 'model_b'
    return 'tie'

train['winner'] = train.apply(get_winner, axis=1)

le = LabelEncoder()
y  = le.fit_transform(train['winner'])
print(f"Clases: {le.classes_}")
print(f"Distribución: {pd.Series(y).value_counts().to_dict()}")

# Texto combinado para TF-IDF
# Concatenamos prompt + response_a + response_b con separadores
train['text_combined'] = (train['prompt_clean'] + ' [SEP] ' +
                          train['response_a_clean'] + ' [SEP] ' +
                          train['response_b_clean'])

test['text_combined']  = (test['prompt_clean'] + ' [SEP] ' +
                          test['response_a_clean'] + ' [SEP] ' +
                          test['response_b_clean'])

print(f"\nEjemplo texto combinado (primeros 300 chars):")
print(train['text_combined'].iloc[0][:300])

Clases: ['model_a' 'model_b' 'tie']
Distribución: {0: 20064, 1: 19652, 2: 17761}

Ejemplo texto combinado (primeros 300 chars):
OK, does pineapple belong on a pizza? Relax and give me fun answer. [SEP] Ah, the age-old culinary conundrum that has divided nations and dinner tables: does pineapple belong on a pizza? The tropical twist of pineapple on pizza, known as Hawaiian pizza, is a hotly debated topic where taste buds batt


In [5]:
print("Nulos en text_combined:")
print(f"  train: {train['text_combined'].isna().sum()}")
print(f"  test:  {test['text_combined'].isna().sum()}")

print("\nNulos por columna limpia:")
for col in ['prompt_clean', 'response_a_clean', 'response_b_clean']:
    print(f"  train[{col}]: {train[col].isna().sum()}")
    print(f"  test[{col}]:  {test[col].isna().sum()}")

Nulos en text_combined:
  train: 91
  test:  0

Nulos por columna limpia:
  train[prompt_clean]: 0
  test[prompt_clean]:  0
  train[response_a_clean]: 67
  test[response_a_clean]:  0
  train[response_b_clean]: 68
  test[response_b_clean]:  0


In [6]:
for df in [train, test]:
    df['prompt_clean']     = df['prompt_clean'].fillna('')
    df['response_a_clean'] = df['response_a_clean'].fillna('')
    df['response_b_clean'] = df['response_b_clean'].fillna('')
    df['text_combined']    = (df['prompt_clean'] + ' [SEP] ' +
                              df['response_a_clean'] + ' [SEP] ' +
                              df['response_b_clean'])

print("NaN imputados — reintentando TF-IDF...")

NaN imputados — reintentando TF-IDF...


In [7]:
# TF-IDF
tfidf = TfidfVectorizer(
    max_features=15_000,
    ngram_range=(1, 1),
    min_df=5,
    sublinear_tf=True
)

X_tfidf_train = tfidf.fit_transform(train['text_combined'])
X_tfidf_test  = tfidf.transform(test['text_combined'])

# Features numéricas
X_num_train = csr_matrix(train[numeric_features].values)
X_num_test  = csr_matrix(test[numeric_features].values)

# Combinar
X_train = hstack([X_tfidf_train, X_num_train])
X_test  = hstack([X_tfidf_test,  X_num_test])

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (57477, 15006)
X_test shape:  (3, 15006)


In [8]:
# Diagnóstico
num_df = train[numeric_features]
print("NaN en features numéricas:")
print(num_df.isna().sum())
print(f"\nInf en features numéricas:")
print(np.isinf(num_df.values).sum(axis=0))

# Fix: reemplazar NaN e Inf
for df in [train, test]:
    df['len_ratio'] = df['len_response_a'] / (df['len_response_b'] + 1)
    for col in numeric_features:
        df[col] = df[col].replace([np.inf, -np.inf], 0).fillna(0)

# Reconstruir matrices
X_num_train = csr_matrix(train[numeric_features].values)
X_num_test  = csr_matrix(test[numeric_features].values)
X_train = hstack([X_tfidf_train, X_num_train])
X_test  = hstack([X_tfidf_test,  X_num_test])

print("\nFix aplicado — matrices reconstruidas")

NaN en features numéricas:
len_prompt         0
len_response_a    67
len_response_b    68
len_diff          91
len_ratio         91
a_is_longer        0
dtype: int64

Inf en features numéricas:
[0 0 0 0 0 0]

Fix aplicado — matrices reconstruidas


In [9]:
# Validación con LogReg liviana
model = LogisticRegression(
    C=1.0,
    max_iter=300,
    solver='lbfgs',
    multi_class='multinomial',
    n_jobs=-1,
    random_state=42
)

groups    = train['prompt']
gkf       = GroupKFold(n_splits=5)
oof_preds = np.zeros((len(train), 3))

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y, groups)):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    model.fit(X_tr, y_tr)
    oof_preds[val_idx] = model.predict_proba(X_val)

    fold_loss = log_loss(y_val, oof_preds[val_idx])
    print(f"Fold {fold+1} — Log Loss: {fold_loss:.4f}")

overall_loss = log_loss(y, oof_preds)
print(f"\nOOF Log Loss: {overall_loss:.4f}")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 1 — Log Loss: 1.0643


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 2 — Log Loss: 1.0624


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 3 — Log Loss: 1.0640


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 4 — Log Loss: 1.0636


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 5 — Log Loss: 1.0641

OOF Log Loss: 1.0637


In [10]:
# Entrenar modelo final sobre todo el train
model.fit(X_train, y)
test_preds = model.predict_proba(X_test)
test_preds = np.clip(test_preds, 1e-7, 1 - 1e-7)
test_preds = test_preds / test_preds.sum(axis=1, keepdims=True)

# Mapear clases al orden correcto
class_order = ['model_a', 'model_b', 'tie']
class_indices = [list(le.classes_).index(c) for c in class_order]

sub['winner_model_a'] = test_preds[:, class_indices[0]]
sub['winner_model_b'] = test_preds[:, class_indices[1]]
sub['winner_tie']     = test_preds[:, class_indices[2]]

sub.to_csv('submission.csv', index=False)
print("Submission guardada")
print(sub)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Submission guardada
        id  winner_model_a  winner_model_b  winner_tie
0   136060        0.264957        0.338866    0.396177
1   211333        0.486244        0.210121    0.303636
2  1233961        0.471809        0.260781    0.267410
